# Ejercicio 11: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [ ]:
!curl -L https://www.allrecipes.com/recipe/45954/roast-sticky-chicken-rotisserie-style/ -o ./rotisserie-chicken.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  386k    0  386k    0     0  1153k      0 --:--:-- --:--:-- --:--:-- 1156k


In [ ]:
from bs4 import BeautifulSoup

file = './rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [ ]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Roast Sticky Chicken-Rotisserie Style'

In [ ]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

4 teaspoons salt
2 teaspoons paprika
1 teaspoon onion powder
1 teaspoon dried thyme
1 teaspoon white pepper
½ teaspoon black pepper
½ teaspoon cayenne pepper
½ teaspoon garlic powder
2 (4 pound) whole chickens
2  onions, quartered


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [ ]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Roast Sticky Chicken-Rotisserie Style
Description: Rotisserie chicken seasoning is a simple homemade spice rub made with basic pantry spices used to make flavorful baked rotisserie chicken at home.
Ingredients:
- 4 teaspoons salt
- 2 teaspoons paprika
- 1 teaspoon onion powder
- 1 teaspoon dried thyme
- 1 teaspoon white pepper
- ½ teaspoon black pepper
- ½ teaspoon cayenne pepper
- ½ teaspoon garlic powder
- 2 (4 pound) whole chickens
- 2  onions, quartered
Instructions:
1. Gather the ingredients.
2. Mix together salt, paprika, onion powder, thyme, white pepper, black pepper, cayenne pepper, and garlic powder in a small bowl; set aside.
3. Remove and discard giblets from chicken. Rinse chicken cavity; pat dry with paper towels. Rub each chicken inside and out with spice mixture. Place 1 onion into the cavity of each chicken. Place chickens in resealable plastic bags or double wrap with plastic wrap. Refrigerate overnight, or at least 4 to 6 hours.
4. Preheat the oven to 2

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [ ]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F45954%2Froast-sticky-chicken-rotisserie-style%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://support.people.inc/hc/en-us/categories/360003648613-Allrecipes
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F45954%2Froast-sticky-chicken-rotisserie-style%2F
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-stews-and-chili/
https://www.allrecipes.com/recipes/16099/everyd

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [ ]:
!pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers

In [ ]:
from langchain_core.documents import Document

recipe_text = f"""
TÍTULO: {title}
DESCRIPCIÓN: {description}

INGREDIENTES:
{', '.join(ingredients)}

INSTRUCCIONES:
{' '.join(instructions)}

NUTRICIÓN:
{', '.join(nutrition_facts)}
"""

docs = [Document(page_content=recipe_text, metadata={"source": "rotisserie-chicken.html"})]

print("Documento creado exitosamente.")

Documento creado exitosamente.


In [ ]:
!pip install langchain-chroma

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# CONFIGURACION DE EMBEDDINGS
print("Cargando modelo de embeddings...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# CREAR BASE DE DATOS VECTORIAL
# Esto guarda tu receta en una base de datos buscable
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

# PROBAR BUSQUEDA (RETRIEVAL)
query = "¿Cuáles son los ingredientes principales del pollo?"

# Buscamos los fragmentos más parecidos a la pregunta
results = vector_db.similarity_search(query, k=1)

Cargando modelo de embeddings...


In [ ]:
print("\n--- RESPUESTA DEL SISTEMA DE BÚSQUEDA ---")
print(f"Información encontrada:\n{results[0].page_content}")


--- RESPUESTA DEL SISTEMA DE BÚSQUEDA ---
Información encontrada:

TÍTULO: Roast Sticky Chicken-Rotisserie Style
DESCRIPCIÓN: Rotisserie chicken seasoning is a simple homemade spice rub made with basic pantry spices used to make flavorful baked rotisserie chicken at home.

INGREDIENTES:
4 teaspoons salt, 2 teaspoons paprika, 1 teaspoon onion powder, 1 teaspoon dried thyme, 1 teaspoon white pepper, ½ teaspoon black pepper, ½ teaspoon cayenne pepper, ½ teaspoon garlic powder, 2 (4 pound) whole chickens, 2  onions, quartered

INSTRUCCIONES:
Gather the ingredients. Mix together salt, paprika, onion powder, thyme, white pepper, black pepper, cayenne pepper, and garlic powder in a small bowl; set aside. Remove and discard giblets from chicken. Rinse chicken cavity; pat dry with paper towels. Rub each chicken inside and out with spice mixture. Place 1 onion into the cavity of each chicken. Place chickens in resealable plastic bags or double wrap with plastic wrap. Refrigerate overnight, or a

In [ ]:
!pip install langchain-google-genai

In [ ]:
# 1. Cambiamos la importacion para usar Google
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# Mi clave principal
mi_clave = #API_KEY

# 2. Configuramos el modelo de Google
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=mi_clave
)

# 3. Ejecutamos la consulta RAG (Esto se mantiene igual)
contexto = results[0].page_content
prompt = f"Usa esta información para responder: {contexto}. Pregunta: {query}"

respuesta = llm.invoke(prompt)

# Google devuelve un objeto mensaje, así que imprimimos el contenido
print(respuesta.content)

[{'type': 'text', 'text': 'Basado en la información proporcionada, los ingredientes principales son:\n\n*   **2 pollos enteros** (de 4 libras cada uno).\n*   **2 cebollas** (cortadas en cuartos).\n\nAdemás, se utiliza una **mezcla de especias** (rub) compuesta por:\n*   Sal\n*   Pimentón (paprika)\n*   Cebolla en polvo\n*   Tomillo seco\n*   Pimienta blanca\n*   Pimienta negra\n*   Pimienta de cayena\n*   Ajo en polvo', 'extras': {'signature': 'EqIKCp8KAXLI2nxbTuQBfyEfddRyL7wjcg/lDqsGVqSqqs34KmPHEIaycTqGNGLtnV9KfL2s0MwpsZzpN8Y3WPB0KBE2WoIKMdstCH+lL7QHUDWpPw84eUivj9hs8Otwnz32XxwMwIbMrUFHJGjRMQEmtUJDfgwNuBx+q+9PaWFU1YBwbLjwLiPSBAtGnTvrYbtW49onpWfrMWL9nDVm8X1JR7w9NjNLgR2NV0JpaSM5Inf9PtgEw6dR91C/GXfeaE1VqLjef9n2z78vnou5i0h88GUbcQ7tBURi3AYLqfICgkjpcF/BcK9/xdIZb1QTlZQWBmtjPl84YftDF13JqdJqU81Ep7ABIjdqoH6C/YICzHk3BRNSGR5/QggRGIMzYBGVY2Hl3UVp+9Y/+4kiRO+BGSlq5/uD23F+lw7UAAQmjbqJLu5N3RwomuQhhfX9oCc1Utm0L7IUQ756pSKBh4NnBD7OyXJ3KfmhFvjMWgXudaXjFzn4iYnu+sUJ0YETC/hbF+KbfhQeAxe4vTugP/uaJ79BzaNIafpg0ml